<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Task Planning Agent </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to build a **Planning Agent** that can break down complex tasks into structured, executable steps and delegate them to worker agents.

This is a core pattern in multi-agent systems: **Task Decomposition + Delegation**.

**Key Topics Covered Today:**
- Task decomposition techniques
- Building a dedicated Planning Agent
- Generating structured task steps
- Passing subtasks to specialized worker agents
- Execution tracking and loop prevention

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall
!pip install -U ddgs
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.

In [2]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Dict
from langsmith import Client

client = Client()

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
react_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt= True)

print("Tools and LLM ready")

Tools and LLM ready


## 2. Creating a Planning Agent

In [3]:
# Planning Agent - Focused on Task Decomposition
planner_prompt = ChatPromptTemplate.from_template(
    """You are a Planning Agent specialized in task decomposition.

User Request: {input}

Break this complex request into **exactly 3 structured steps**:
1. Research Step (what data needs to be collected)
2. Calculation Step (what needs to be calculated)
3. Summary Step (what final output is expected)

Return ONLY a numbered list with these 3 steps. Keep each step short and clear."""
)

planner_chain = planner_prompt | llm

print("Planning Agent created - focused on task decomposition")

Planning Agent created - focused on task decomposition


## 3. Worker Agents

In [8]:
# Researcher Agent
researcher_prompt = react_prompt.partial(
    system_message="""You are a Researcher Agent.
Your job is to collect the required data using Web_Search.
Return the information in a clear, simple format."""
)

researcher_agent = create_react_agent(llm=llm, tools=[search_tool], prompt=researcher_prompt)
researcher_executor = AgentExecutor(
    agent=researcher_agent,
    tools=[search_tool],
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

# Calculator Agent - Simple and stable
calculator_prompt = react_prompt.partial(
    system_message="""You are a Calculator Agent.
Your job is to take the research data and perform the required calculations using the calculate_math tool.
Return a short, clean summary of the results."""
)

calculator_agent = create_react_agent(llm=llm, tools=[calculate_math], prompt=calculator_prompt)
calculator_executor = AgentExecutor(
    agent=calculator_agent,
    tools=[calculate_math],
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Worker Agents (Researcher & Calculator) created")

Worker Agents (Researcher & Calculator) created


## 4. Demo: Task Planning + Delegation

In [9]:
def run_planning_agent_demo():
    user_query = (
        "Estimate a project quote for building an end-to-end AI agentic system " ## end-to-end AI agentic system
        "requiring 80 estimated hours. Factor in standard Saudi AI Engineer hourly rates in SAR, " ## 80 estimated hours
        "a 20% safety buffer, and 15% estimated Zakat/tax reserve. Calculate the final "
        "fixed project price in SAR and effective net hourly profit."
    )

    print("=== Task Planning Agent Demo: AI Project Pricing (KSA) ===\n")

    # Step 1: Planning Agent decomposes the task
    print("1. Planning Agent - Task Decomposition")
    plan_result = planner_chain.invoke({"input": user_query})
    plan_text = plan_result.content if hasattr(plan_result, "content") else str(plan_result)
    print(plan_text)
    print("\n" + "-" * 60 + "\n")

    # Step 2: Researcher Agent executes research step
    print("2. Researcher Agent - Executing Research Step")
    research_input = (
        "Find average hourly benchmark rates in SAR for a freelance AI Engineer "
        "or Machine Learning specialist in Saudi Arabia."
    )
    research_result = researcher_executor.invoke({"input": research_input})
    research_data = research_result["output"]
    print("Research Output:")
    print(research_data)
    print("\n" + "-" * 60 + "\n")

    # Step 3: Calculator Agent executes calculation step
    print("3. Calculator Agent - Executing Calculation Step")
    calc_input = f"""Using this research data:
{research_data}

Calculate using SAR:
1. Total billable hours including a 20% safety buffer on the base 80 hours.
2. Total gross project price in SAR based on the benchmark hourly rate.
3. Net profit in SAR after deducting 15% for Zakat/taxes from the total gross price.
4. Effective net hourly profit in SAR earned per base hour."""

    calc_result = calculator_executor.invoke({"input": calc_input})
    print("Calculation Output:")
    print(calc_result["output"])
    print("\n" + "=" * 70)
    print("DEMO COMPLETE: AI Engineering project pricing pipeline executed.")

run_planning_demo = run_planning_agent_demo
run_planning_demo()

=== Task Planning Agent Demo: AI Project Pricing (KSA) ===

1. Planning Agent - Task Decomposition
1. **Research Step**: Collect data on the standard hourly rates for AI Engineers in Saudi Arabia (in SAR) and any relevant tax/Zakat rates applicable to the project.

2. **Calculation Step**: Calculate the total project cost by multiplying the hourly rate by 80 hours, adding a 20% safety buffer, and then including a 15% Zakat/tax reserve on the total.

3. **Summary Step**: Present the final fixed project price in SAR and determine the effective net hourly profit by dividing the total project price by 80 hours and subtracting the hourly rate.

------------------------------------------------------------

2. Researcher Agent - Executing Research Step
Research Output:
The average hourly benchmark rate for a freelance AI Engineer or Machine Learning specialist in Saudi Arabia is approximately 102.21 ر.س., with freelance rates potentially varying widely based on experience and project specific

In [ ]:
# def run_planning_agent_demo():
#     user_query = "Plan a budget trip from Riyadh to Istanbul for 7 days. Calculate the total cost in SAR and the percentage that flights represent of the total budget."

#     print("=== Task Planning Agent Demo ===\n")

#     # Step 1: Planning Agent decomposes the task
#     print("1. Planning Agent - Task Decomposition")
#     plan_result = planner_chain.invoke({"input": user_query})
#     plan_text = plan_result.content if hasattr(plan_result, "content") else str(plan_result)

#     print(plan_text)
#     print("\n" + "-" * 60 + "\n")

#     # Step 2: Researcher Agent executes research step
#     print("2. Researcher Agent - Executing Research Step")
#     research_input = "Find the cheapest round-trip flight price from Riyadh to Istanbul in USD, average hotel price per night in Istanbul in USD, and current USD to SAR exchange rate."
#     research_result = researcher_executor.invoke({"input": research_input})
#     research_data = research_result["output"]

#     print("Research Output:")
#     print(research_data)
#     print("\n" + "-" * 60 + "\n")

#     # Step 3: Calculator Agent executes calculation step
#     print("3. Calculator Agent - Executing Calculation Step")
#     calc_input = f"Using this research data:\n{research_data}\n\nCalculate the total trip cost in SAR for round-trip flight + 7 nights hotel, and the percentage of the total cost that comes from the flight."
#     calc_result = calculator_executor.invoke({"input": calc_input})

#     print("Calculation Output:")
#     print(calc_result["output"])

#     print("\n" + "="*70)
#     print("DEMO COMPLETE: Complex task was successfully decomposed and executed by specialized agents.")

# run_planning_demo = run_planning_agent_demo
# run_planning_demo()

=== Task Planning Agent Demo ===

1. Planning Agent - Task Decomposition
1. **Research Step**: Collect data on flight prices from Riyadh to Istanbul, accommodation options for 7 nights, daily food expenses, local transportation costs, and any attraction entry fees.

2. **Calculation Step**: Calculate the total cost by summing the flight costs, accommodation, food, transportation, and attractions. Determine the percentage of the total budget that flights represent.

3. **Summary Step**: Present the total trip cost in SAR and the percentage of the total budget attributed to flights.

------------------------------------------------------------

2. Researcher Agent - Executing Research Step
Research Output:
Agent stopped due to iteration limit or time limit.

------------------------------------------------------------

3. Calculator Agent - Executing Calculation Step
Calculation Output:
Agent stopped due to iteration limit or time limit.

DEMO COMPLETE: Complex task was successfully deco

## 5. Reflection Questions

1. How does the Planning Agent help prevent infinite loops?
2. What are the benefits of generating a structured plan before execution?
3. How did we delegate tasks to specialized worker agents?
4. What challenges might occur if the plan is too vague or too long?
5. How would you improve this system to make delegation more dynamic?
